In [ ]:
IS_TRAIN = True          
IS_INFERENCE = True 

In [ ]:
!pip install -q transformers==4.44.* --force-reinstall

In [ ]:
!pip uninstall numpy -y

In [ ]:
!pip install -q numpy==1.26.0 -y

In [ ]:
!pip install -q packaging
!pip install -q ninja
!pip install -q flash_attn
!pip install -q datasets
!pip install -q timm
!pip install -q einops
!pip install -q peft
!pip install -q deepspeed
!pip install -q bitsandbytes
!pip install -q decord
!pip install -q gdown

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id='5CD-AI/Vintern-1B-v2',
    local_dir='/kaggle/working/Vintern/pretrained/Vintern-1B-v2',
    local_dir_use_symlinks=False,
    resume_download=True,
)

In [ ]:
from pathlib import Path

INPUT          = Path("/kaggle/input/datasets/maituananh511/work-dir-finetune-44761")
KAGGLE_WORKING = Path("/kaggle/working")

LORA_CKPT_DIR  = Path("/kaggle/input/datasets/maituananh511/work-dir-finetune-44761/work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa/checkpoint-44671")# Merge output -> WORKING (writable)
LORA_MERGE_DIR = KAGGLE_WORKING / "work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa_merge"

PRETRAIN_DIR   = KAGGLE_WORKING / "Vintern/pretrained/Vintern-1B-v2"

VINTERN_CHAT   = INPUT / "/kaggle/input/datasets/maituananh511/vintern/Vintern-main/internvl_chat"

FOLDER_IMAGES        = Path("/kaggle/input/datasets/maituananh511/viet-chart-vqa-images/output/Viet-Chart-VQA-images")
VIETNAMESE_DATA_PATH = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")

print("INPUT          exists:", INPUT.exists())
print("LORA_CKPT_DIR  exists:", LORA_CKPT_DIR.exists())
print("LORA_MERGE_DIR exists:", LORA_MERGE_DIR.exists())
print("PRETRAIN_DIR   exists:", PRETRAIN_DIR.exists())
print("VINTERN_CHAT   exists:", VINTERN_CHAT.exists())
print("FOLDER_IMAGES  exists:", FOLDER_IMAGES.exists())
print("VIETNAMESE_DATA exists:", VIETNAMESE_DATA_PATH.exists())


In [ ]:
from datasets import load_from_disk, concatenate_datasets, Dataset
import json, shutil, os
from PIL import Image
import pyarrow as pa

SRC_DATASET = "/kaggle/input/datasets/maituananh511/test-dataset-chart-vqa/vi_chart_dataset"
DST_DATASET = "/kaggle/working/vi_chart_dataset"

if not os.path.exists(DST_DATASET):
    print("Copying dataset to working dir...")
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print("Done.")
else:
    print("Dataset already copied, skipping.")

vi_chart_dataset = load_from_disk(DST_DATASET)
print(vi_chart_dataset)


In [ ]:
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))

print(f"Vietnamese dataset: {len(vietnamese_records)} records loaded")


In [ ]:
def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        content = str(turn.get('content') or turn.get('value', ''))
        return {'role': role, 'content': content}
    return {'role': 'assistant', 'content': str(turn)}

def align_conversations_schema(dataset, reference_dataset):
    ref_conv_type  = reference_dataset.data.schema.field('conversations').type
    ref_struct_type = ref_conv_type.value_type
    field_order = [ref_struct_type.field(i).name for i in range(ref_struct_type.num_fields)]
    pa_table   = dataset.data.table
    conv_arr   = pa_table.column('conversations').combine_chunks()
    struct_arr = conv_arr.values
    arrays = [struct_arr.field(f) for f in field_order]
    fields = [pa.field(f, pa.string()) for f in field_order]
    new_struct = pa.StructArray.from_arrays(arrays, fields=fields)
    new_conv   = pa.ListArray.from_arrays(conv_arr.offsets, new_struct)
    idx = pa_table.schema.get_field_index('conversations')
    new_table = pa_table.set_column(idx, 'conversations', new_conv)
    return Dataset(new_table)

vn_rows = []
for record in vietnamese_records:
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Warning: cannot open {img_path}: {e}")
        continue
    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs) - 1, 2)]
    for idx, (q, a) in enumerate(pairs):
        record_id = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_rows.append({'id': record_id, 'image': image, 'conversations': [q, a]})

TEST_SIZE = 200
vn_train_rows = vn_rows[:-TEST_SIZE]
vn_test_rows  = vn_rows[-TEST_SIZE:]

vi_vietnamese_train = Dataset.from_list(vn_train_rows)
vi_vietnamese_test  = Dataset.from_list(vn_test_rows)
vi_vietnamese_train = align_conversations_schema(vi_vietnamese_train, vi_chart_dataset['train'])
vi_vietnamese_test  = align_conversations_schema(vi_vietnamese_test,  vi_chart_dataset['test'])

vi_chart_30k   = vi_chart_dataset['train'].shuffle(seed=42).select(range(30000))
merged_train   = concatenate_datasets([vi_chart_30k, vi_vietnamese_train])
merged_test    = concatenate_datasets([vi_chart_dataset['test'], vi_vietnamese_test])
vi_chart_dataset['train'] = merged_train
vi_chart_dataset['test']  = merged_test

print(f"Total train: {len(vi_chart_dataset['train'])}")
print(f"Total test : {len(vi_chart_dataset['test'])}")

In [ ]:
import shutil, os

paths = [
    '/kaggle/working/work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa_merge',
    '/kaggle/working/merge_lora.py'
]

for p in paths:
    if os.path.exists(p):
        if os.path.isdir(p):
            shutil.rmtree(p)
        else:
            os.remove(p)
        print(f'Deleted: {p}')
    else:
        print(f'Not found: {p}')

In [ ]:
import torch

torch.cuda.empty_cache() 

In [ ]:
print("Loading base model...")
import sys
sys.path.append("/kaggle/input/datasets/maituananh511/vintern/Vintern-main/internvl_chat")
from internvl.model.internvl_chat import InternVLChatModel
from transformers import AutoModel, AutoTokenizer

model = InternVLChatModel.from_pretrained(
    str(LORA_CKPT_DIR),
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(str(LORA_CKPT_DIR), trust_remote_code=True)

print("Merging LoRA...")
if model.config.use_llm_lora:
    model.language_model.merge_and_unload()
    model.language_model = model.language_model.model
    model.config.use_llm_lora = 0

if model.config.use_backbone_lora:
    model.vision_model.merge_and_unload()
    model.vision_model = model.vision_model.model
    model.config.use_backbone_lora = 0

print("Saving merged model...")
model.save_pretrained(str(LORA_MERGE_DIR))
tokenizer.save_pretrained(str(LORA_MERGE_DIR))

for pyf in PRETRAIN_DIR.glob("*.py"):
    shutil.copy2(str(pyf), str(LORA_MERGE_DIR))

print("Done!", LORA_MERGE_DIR)

In [ ]:
import json
from pathlib import Path

cfg_path = Path("/kaggle/input/datasets/maituananh511/work-dir-finetune-44761/work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa/checkpoint-44671/config.json")
cfg = json.load(open(cfg_path))
print("use_llm_lora     :", cfg.get("use_llm_lora"))
print("use_backbone_lora:", cfg.get("use_backbone_lora"))

In [ ]:
import numpy as np
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
import matplotlib.pyplot as plt
from tqdm import tqdm

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width  = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width  // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width  // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size)))
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB') if isinstance(image_file, str) else image_file
    image = image.resize((input_size, input_size))
    transform = build_transform(input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = torch.stack([transform(img) for img in images])
    return pixel_values

In [ ]:
from transformers import AutoModel, AutoTokenizer

print("Loading PRETRAIN model...")
pretrain_model = AutoModel.from_pretrained(
    PRETRAIN_DIR,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).eval().cuda()

tokenizer = AutoTokenizer.from_pretrained(PRETRAIN_DIR, trust_remote_code=True, use_fast=False)
print("Pretrain model loaded.")

print("\nLoading FINE-TUNED (LoRA merged) model...")
lora_model = AutoModel.from_pretrained(
    LORA_MERGE_DIR,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True
).eval().cuda()
print("Fine-tuned model loaded.")

In [ ]:
generation_config = dict(max_new_tokens=1024, do_sample=False, num_beams=3, repetition_penalty=2.0)

for i in range(min(5, len(vi_chart_dataset['test']))):
    item         = vi_chart_dataset['test'][i]
    image        = item['image']
    question_txt = str(item['conversations'][0]['content'])
    ground_truth = str(item['conversations'][1]['content'])
    question     = f'<image>\n{question_txt}'

    pixel_values = load_image(image, max_num=12).to(torch.bfloat16).cuda()

    resp_pretrain  = pretrain_model.chat(tokenizer, pixel_values, question, generation_config)
    resp_finetune  = lora_model.chat(tokenizer, pixel_values, question, generation_config)

    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f"Sample {i+1}")
    plt.show()

    print(f" Question   : {question_txt}")
    print(f" Ground Truth: {ground_truth}")
    print(f" Pretrain    : {resp_pretrain}")
    print(f" Fine-tuned  : {resp_finetune}")
    print("="*70)

In [ ]:
import re
import matplotlib.pyplot as plt

log_path = INPUT / "work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa/training_log.txt"
pattern  = r"{'loss': (\d+\.\d+), 'grad_norm': .*, 'learning_rate': .*, 'epoch': (\d+\.\d+)}"

epochs, losses = [], []
try:
    with open(log_path, 'r') as f:
        for line in f:
            m = re.search(pattern, line)
            if m:
                losses.append(float(m.group(1)))
                epochs.append(float(m.group(2)))
except FileNotFoundError:
    print(f'Log not found: {log_path}')

if losses:
    window = min(50, len(losses))
    smoothed = [sum(losses[max(0,i-window):i+1]) / len(losses[max(0,i-window):i+1]) for i in range(len(losses))]

    plt.figure(figsize=(14, 5))
    plt.plot(losses,   color='steelblue', alpha=0.4, label='Raw loss')
    plt.plot(smoothed, color='red', linewidth=2, label='Smoothed loss')
    plt.xlabel('Logging step')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(KAGGLE_WORKING / 'loss_plot.png', dpi=150)
    plt.show()
    print('Saved: loss_plot.png')
else:
    print('Không tìm thấy loss data trong log.')


In [ ]:
!pip install -q rouge_score
!pip install -q evaluate 
!pip install -q underthesea
!pip install -q bert-score 
!pip install --upgrade nltk

In [ ]:
!pip install -q numpy==2.0.0

In [ ]:
import importlib, sys

for key in list(sys.modules.keys()):
    if 'nltk' in key:
        del sys.modules[key]

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
print('OK')

In [ ]:
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch, nltk, os
from underthesea import word_tokenize

def setup_nltk_data(nltk_data_path='/usr/share/nltk_data'):
    os.makedirs(nltk_data_path, exist_ok=True)
    nltk.data.path.append(nltk_data_path)
    for pkg in ['punkt', 'wordnet', 'omw-1.4']:
        nltk.download(pkg, download_dir=nltk_data_path, quiet=True)

def compute_bertscore(responses, references):
    """Tính BERTScore theo batch, tránh import ở top-level"""
    try:
        import bert_score as bs_lib
        _, _, F1 = bs_lib.score(responses, references, lang='vi', verbose=False,
                                rescale_with_baseline=False)
        return F1.tolist()
    except Exception as e:
        print(f'BERTScore batch failed: {e}')
        return [0.0] * len(responses)

def evaluate_model(dataset, model, tokenizer, model_name='model', debug=False):
    setup_nltk_data()

    if not dataset or 'test' not in dataset:
        raise ValueError("Invalid dataset: 'test' key not found")

    bleu_scores, meteor_scores = [], []
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    all_responses, all_references = [], []
    results = []

    scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method1
    gen_cfg  = dict(max_new_tokens=1024, do_sample=False, num_beams=3, repetition_penalty=2.0)

    for i in tqdm(range(len(dataset['test'])), desc=f'Evaluating [{model_name}]'):
        item = dataset['test'][i]
        if not all(k in item for k in ['id', 'image', 'conversations']):
            continue

        image_id     = item['id']
        image        = item['image']
        question     = '<image>\n' + str(item['conversations'][0]['content'])
        ground_truth = str(item['conversations'][1]['content'])

        try:
            pixel_values = load_image(image, max_num=12).to(torch.bfloat16).cuda()
        except Exception as e:
            if debug: print(f'Error loading image {i+1}: {e}')
            continue

        response = model.chat(tokenizer, pixel_values, question, gen_cfg)

        ref_tok  = word_tokenize(ground_truth, format='text').split()
        resp_tok = word_tokenize(response,     format='text').split()

        try:
            bleu = sentence_bleu([ref_tok], resp_tok, weights=(0.25,)*4, smoothing_function=smoothie)
        except ZeroDivisionError:
            bleu = 0.0
        bleu_scores.append(bleu)

        try:
            meteor = nltk_meteor([ref_tok], resp_tok)
        except Exception:
            meteor = 0.0
        meteor_scores.append(meteor)

        rouge = scorer.score(ground_truth, response)
        rouge_scores['rouge1'].append(rouge['rouge1'].fmeasure)
        rouge_scores['rouge2'].append(rouge['rouge2'].fmeasure)
        rouge_scores['rougeL'].append(rouge['rougeL'].fmeasure)

        all_responses.append(response)
        all_references.append(ground_truth)

        results.append({
            'id': image_id, 'question': question,
            'ground_truth': ground_truth, 'response': response,
            'bleu': bleu, 'meteor': meteor,
            'rouge1': rouge['rouge1'].fmeasure,
            'rouge2': rouge['rouge2'].fmeasure,
            'rougeL': rouge['rougeL'].fmeasure,
            'bertscore': 0.0  # placeholder, fill sau
        })

        if debug:
            print(f'Sample {i+1} | Pred: {response[:80]} | GT: {ground_truth[:80]}')

    print(f'Computing BERTScore for {len(all_responses)} samples...')
    bert_scores = compute_bertscore(all_responses, all_references)
    for j, row in enumerate(results):
        row['bertscore'] = bert_scores[j]

    n = len(results)
    avg_scores = {
        'bleu':      sum(bleu_scores)           / n if n else 0.0,
        'meteor':    sum(meteor_scores)          / n if n else 0.0,
        'rouge1':    sum(rouge_scores['rouge1']) / n if n else 0.0,
        'rouge2':    sum(rouge_scores['rouge2']) / n if n else 0.0,
        'rougeL':    sum(rouge_scores['rougeL']) / n if n else 0.0,
        'bertscore': sum(bert_scores)            / n if n else 0.0,
    }

    print(f'[{model_name}] Average scores:')
    for k, v in avg_scores.items():
        print(f'  {k:12s}: {v:.4f}')

    return pd.DataFrame(results), avg_scores

print('evaluate_model defined ')

In [ ]:
def calculate_score_differences(pretrain_scores, finetune_scores):
    differences = {}
    metrics = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']  
    
    for metric in metrics:
        if metric in pretrain_scores and metric in finetune_scores:
            differences[metric] = finetune_scores[metric] - pretrain_scores[metric]
        else:
            differences[metric] = 0.0
            print(f"Warning: Metric {metric} not found in one or both score dictionaries.")
    
    return differences

In [ ]:
if IS_INFERENCE: 
    pretrain_df, pretrain_avg_scores = evaluate_model(vi_chart_dataset, pretrain_model, tokenizer, debug=False)
    print("Average Scores:", pretrain_avg_scores)

In [ ]:
if IS_INFERENCE: 
    fine_tune_df, fine_tune_avg_scores = evaluate_model(vi_chart_dataset, lora_model, tokenizer, debug=False)
    print("Average Scores:", fine_tune_avg_scores)

In [ ]:
if IS_INFERENCE: 
    differences = calculate_score_differences(pretrain_avg_scores, fine_tune_avg_scores)
    
    print("Score Differences (Fine-tune - Pretrain):")
    for metric, diff in differences.items():
        print(f"{metric.upper()}: {diff:.4f}")

In [ ]:
if IS_INFERENCE:
    data = {
        'Metric': ['BLEU', 'METEOR', 'ROUGE1', 'ROUGE2', 'ROUGEL', 'BERTScore'],  
        'Pretrain': [pretrain_avg_scores['bleu'], pretrain_avg_scores['meteor'], 
                     pretrain_avg_scores['rouge1'], pretrain_avg_scores['rouge2'], 
                     pretrain_avg_scores['rougeL'], pretrain_avg_scores['bertscore']],
        'Fine-tune': [fine_tune_avg_scores['bleu'], fine_tune_avg_scores['meteor'], 
                      fine_tune_avg_scores['rouge1'], fine_tune_avg_scores['rouge2'], 
                      fine_tune_avg_scores['rougeL'], fine_tune_avg_scores['bertscore']],
        'Difference': [differences['bleu'], differences['meteor'], 
                       differences['rouge1'], differences['rouge2'], 
                       differences['rougeL'], differences['bertscore']]
    }
    
    compare_df = pd.DataFrame(data)
    
    compare_df[['Pretrain', 'Fine-tune', 'Difference']] = compare_df[['Pretrain', 'Fine-tune', 'Difference']].round(4)
    
    compare_df

In [ ]:
if IS_INFERENCE:
    pretrain_df = pretrain_df.rename(columns={
        'response': 'response_pretrain',
        'bleu': 'bleu_pretrain',
        'meteor': 'meteor_pretrain',
        'rouge1': 'rouge1_pretrain',
        'rouge2': 'rouge2_pretrain',
        'rougeL': 'rougeL_pretrain',
        'bertscore': 'bertscore_pretrain'  
    })
    
    fine_tune_df = fine_tune_df.rename(columns={
        'response': 'response_finetune',
        'bleu': 'bleu_finetune',
        'meteor': 'meteor_finetune',
        'rouge1': 'rouge1_finetune',
        'rouge2': 'rouge2_finetune',
        'rougeL': 'rougeL_finetune',
        'bertscore': 'bertscore_finetune'  
    })
    
    merged_df = pd.merge(
        pretrain_df,
        fine_tune_df,
        on=['id', 'question', 'ground_truth'],
        how='inner'
    )
    
    merged_columns = [
        'id', 'question', 'ground_truth',
        'response_pretrain', 'response_finetune',
        'bleu_pretrain', 'bleu_finetune',
        'meteor_pretrain', 'meteor_finetune',
        'rouge1_pretrain', 'rouge1_finetune',
        'rouge2_pretrain', 'rouge2_finetune',
        'rougeL_pretrain', 'rougeL_finetune',
        'bertscore_pretrain', 'bertscore_finetune'  
    ]
    
    merged_df = merged_df[merged_columns]
    merged_df.to_csv('/kaggle/working/merged_evaluation_results.csv', index=False, encoding='utf-8')
    merged_df.head(5)

In [ ]:
import shutil, os

dirs_to_delete = [
    '/kaggle/working/Vintern',
    '/kaggle/working/cache',
    '/kaggle/working/internvl_chat',
    '/kaggle/working/internvl_patched',
]

for d in dirs_to_delete:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'Deleted: {d}')
    else:
        print(f'Not found: {d}')

In [ ]:
import pandas as pd

metrics = ['bleu','meteor','rouge1','rouge2','rougeL','bertscore']
compare_df = pd.DataFrame({
    'Metric':     [m.upper() for m in metrics],
    'Pretrain':   [round(pretrain_avg_scores[m], 4) for m in metrics],
    'Fine-tuned': [round(fine_tune_avg_scores[m], 4) for m in metrics],
    'Difference': [round(fine_tune_avg_scores[m]-pretrain_avg_scores[m], 4) for m in metrics],
})

def highlight_diff(val):
    if isinstance(val, float):
        return 'color: green' if val > 0 else ('color: red' if val < 0 else '')
    return ''

display(compare_df.style.applymap(highlight_diff, subset=['Difference']))


In [ ]:
x = range(len(metrics)); width = 0.35
fig, ax = plt.subplots(figsize=(12,5))
b1 = ax.bar([i-width/2 for i in x], compare_df['Pretrain'],   width, label='Pretrain',   color='steelblue')
b2 = ax.bar([i+width/2 for i in x], compare_df['Fine-tuned'], width, label='Fine-tuned', color='mediumseagreen')
ax.set_xticks(list(x)); ax.set_xticklabels(compare_df['Metric'])
ax.set_ylabel('Score'); ax.set_title('Pretrain vs Fine-tuned — Metric Comparison')
ax.legend(); ax.bar_label(b1, fmt='%.4f', padding=2, fontsize=8); ax.bar_label(b2, fmt='%.4f', padding=2, fontsize=8)
plt.tight_layout()
plt.savefig(KAGGLE_WORKING/'metric_comparison.png', dpi=150); plt.show()
print('Saved: metric_comparison.png')


In [ ]:
import os, shutil, zipfile
from pathlib import Path
from datetime import datetime
from IPython.display import Javascript, display

timestamp  = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_DIR = Path(f'/kaggle/working/export_Vintern_1B_v2_{timestamp}')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if LORA_MERGE_DIR.exists():
    shutil.copytree(str(LORA_MERGE_DIR), str(EXPORT_DIR / 'model_merged'))
    print('Copied merged model')
else:
    print(f'Merged model not found: {LORA_MERGE_DIR}')

if LORA_CKPT_DIR.exists():
    shutil.copytree(str(LORA_CKPT_DIR), str(EXPORT_DIR / 'lora_checkpoint' / LORA_CKPT_DIR.name))
    print(f'Copied LoRA checkpoint: {LORA_CKPT_DIR.name}')

log_file = INPUT / 'work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa/training_log.txt'
if log_file.exists():
    shutil.copy2(str(log_file), str(EXPORT_DIR / 'training_log.txt'))
    print('Copied training_log.txt')

for fname in ['loss_plot.png', 'metric_comparison.png']:
    p = KAGGLE_WORKING / fname
    if p.exists():
        shutil.copy2(str(p), str(EXPORT_DIR / fname))
        print(f'Copied {fname}')

eval_csv = KAGGLE_WORKING / 'merged_evaluation_results.csv'
if eval_csv.exists():
    shutil.copy2(str(eval_csv), str(EXPORT_DIR / 'merged_evaluation_results.csv'))
    print('Copied merged_evaluation_results.csv')

sample_imgs = sorted(KAGGLE_WORKING.glob('sample_*.png'))
if sample_imgs:
    sample_dir = EXPORT_DIR / 'sample_inference_images'
    sample_dir.mkdir(exist_ok=True)
    for img in sample_imgs:
        shutil.copy2(str(img), str(sample_dir / img.name))
    print(f'Copied {len(sample_imgs)} sample inference image(s)')

ZIP_PATH = KAGGLE_WORKING / f'Vintern_1B_v2_outputs_{timestamp}.zip'
print(f'\nZipping...')
with zipfile.ZipFile(str(ZIP_PATH), 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(str(EXPORT_DIR)):
        for file in files:
            abs_path = os.path.join(root, file)
            arc_name = os.path.relpath(abs_path, str(EXPORT_DIR))
            zf.write(abs_path, arc_name)

zip_mb = ZIP_PATH.stat().st_size / (1024*1024)
print(f'ZIP created: {ZIP_PATH.name}  ({zip_mb:.1f} MB)')

print('\nNội dung ZIP:')
for root, _, files in os.walk(str(EXPORT_DIR)):
    level = root.replace(str(EXPORT_DIR), '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root) or "export"}/')
    for f in files:
        fsize = os.path.getsize(os.path.join(root, f)) / 1024
        print(f'{"  "*(level+1)}{f}  ({fsize:.0f} KB)')
